# Camada Silver — Movimentos

Lê a Bronze, limpa e estrutura os dados:
- Achata o struct `_source` em colunas planas
- Explode o array `movimentos` (1 linha por evento processual)
- Parseia o número CNJ em segmentos separados
- Calcula campos temporais com window functions
- Grava em `judicial.silver.movimentos` via MERGE incremental

**Fonte:** `judicial.bronze.datajud_raw`  
**Destino:** `judicial.silver.movimentos`

> Sem DataFrame API — todas as transformações em `spark.sql()` puro.

## 1. Configuração

In [0]:
BRONZE_TABLE = "judicial.bronze.datajud_raw"
SILVER_TABLE = "judicial.silver.movimentos"


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS judicial.silver;

## 2. Extração e explosão da Bronze

A Bronze guarda cada hit do Elasticsearch com `_source` como struct aninhado.
Aqui extraímos os campos relevantes em colunas planas e explodimos
o array `movimentos` — cada evento processual vira uma linha independente.

**`LATERAL VIEW EXPLODE`** é o equivalente SQL do `explode()` do PySpark:
para cada processo com N movimentos, gera N linhas com os dados do processo repetidos.
https://docs.databricks.com/aws/pt/sql/language-manual/

https://docs.databricks.com/aws/pt/sql/language-manual/sql-ref-syntax-qry-select-lateral-view

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_movimentos_raw AS
    SELECT
        _source.numeroProcesso                        AS numero_processo,
        _source.tribunal                              AS sigla_tribunal,
        _source.grau                                  AS grau,
        to_timestamp(_source.dataAjuizamento)         AS data_ajuizamento,
        _source.classe.codigo                         AS codigo_classe,
        _source.classe.nome                           AS nome_classe,
        _source.orgaoJulgador.codigo                  AS codigo_orgao,
        _source.orgaoJulgador.nome                    AS nome_orgao,
        _source.nivelSigilo                           AS nivel_sigilo,
        mov.codigo                                    AS codigo_movimento,
        mov.nome                                      AS nome_movimento,
        to_timestamp(mov.dataHora)                    AS data_movimento
    FROM {BRONZE_TABLE}
    LATERAL VIEW EXPLODE(_source.movimentos) t AS mov
""")

total = spark.sql("SELECT COUNT(*) AS n FROM vw_movimentos_raw").collect()[0]["n"]
print(f"Movimentos extraídos: {total:,}")


## 3. Parse do número CNJ

A API retorna o número como 20 dígitos consecutivos, sem separadores.  
Formato: `NNNNNNN DD AAAA J TT OOOO`

| Segmento | Dígitos | Significado |
|---|---|---|
| `NNNNNNN` | 7 | Número sequencial no órgão |
| `DD` | 2 | Dígitos verificadores |
| `AAAA` | 4 | Ano de ajuizamento |
| `J` | 1 | Segmento de justiça (5=TJ, 4=JF, 6=TRT…) |
| `TT` | 2 | Código do tribunal |
| `OOOO` | 4 | Código do órgão / vara de origem |

Exemplo: `00002464420248160192` → `0000246` `44` `2024` `8` `16` `0192`

In [0]:
CNJ_PATTERN = r'(\d{7})(\d{2})(\d{4})(\d)(\d{2})(\d{4})'

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_movimentos_parsed AS
    SELECT
        *,
        regexp_extract(numero_processo, '{CNJ_PATTERN}', 1)       AS cnj_sequencial,
        regexp_extract(numero_processo, '{CNJ_PATTERN}', 2)       AS cnj_digitos,
        cast(regexp_extract(numero_processo, '{CNJ_PATTERN}', 3) AS int) AS cnj_ano,
        cast(regexp_extract(numero_processo, '{CNJ_PATTERN}', 4) AS int) AS cnj_justica,
        cast(regexp_extract(numero_processo, '{CNJ_PATTERN}', 5) AS int) AS cnj_cod_tribunal,
        regexp_extract(numero_processo, '{CNJ_PATTERN}', 6)       AS cnj_origem
    FROM vw_movimentos_raw
""")

print("Parse do número CNJ aplicado")


## 4. Window functions — campos temporais

Window functions operam sobre uma "janela" de linhas relacionadas — aqui, todos os
movimentos do mesmo processo, ordenados por data.

- `LAG(data_movimento)` devolve a data do movimento anterior na mesma janela
- `datediff` calcula a diferença em dias entre duas datas  
- `ROW_NUMBER() DESC = 1` marca o movimento mais recente de cada processo

https://docs.databricks.com/aws/pt/sql/language-manual/sql-ref-functions-builtin

https://docs.databricks.com/aws/pt/sql/language-manual/sql-ref-window-functions

In [0]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_silver_final AS
    SELECT
        *,
        datediff(data_movimento, data_ajuizamento)          AS dias_desde_ajuizamento,
        datediff(data_movimento, lag(data_movimento) OVER (w_asc))
                                                            AS dias_desde_ultimo_movimento,
        row_number() OVER (w_desc) = 1                        AS is_ultimo_movimento,
        current_timestamp()                                 AS data_processamento
    FROM vw_movimentos_parsed
    WINDOW
        w_asc  AS (PARTITION BY numero_processo ORDER BY data_movimento),
        w_desc AS (PARTITION BY numero_processo ORDER BY data_movimento DESC)
""")

print("Window functions calculadas")

## 5. MERGE incremental na Silver

A chave de match é `(numero_processo, codigo_movimento, data_movimento)`:  
identifica univocamente um evento processual.

- `WHEN MATCHED` — atualiza o registro se algum campo mudou (ex: `is_ultimo_movimento` virou `false`)
- `WHEN NOT MATCHED` — insere movimentos novos que ainda não existem na Silver

In [0]:
tabela_existe = spark.catalog.tableExists(SILVER_TABLE)

if not tabela_existe:
    spark.sql(f"""
        CREATE TABLE {SILVER_TABLE}
        USING DELTA
        AS SELECT * FROM vw_silver_final
    """)
    total = spark.sql(f"SELECT COUNT(*) AS n FROM {SILVER_TABLE}").collect()[0]["n"]
    print(f"Tabela criada: {total:,} movimentos")

else:
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS t
        USING vw_silver_final AS s
        ON  t.numero_processo  = s.numero_processo
        AND t.codigo_movimento = s.codigo_movimento
        AND t.data_movimento   = s.data_movimento
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    total = spark.sql(f"SELECT COUNT(*) AS n FROM {SILVER_TABLE}").collect()[0]["n"]
    print(f"MERGE concluído — {total:,} movimentos na Silver")

## 6. Verificação

In [0]:
# Contagem total vs chave única — se iguais, sem duplicatas
spark.sql("""
    SELECT
        COUNT(*)                                                                              AS total_movimentos,
        COUNT(DISTINCT numero_processo)                                                       AS processos_unicos,
        COUNT(DISTINCT CONCAT(numero_processo, codigo_movimento, CAST(data_movimento AS STRING))) AS chave_unica
    FROM judicial.silver.movimentos
""").show()


In [0]:
# Valida que dias_desde_ajuizamento nunca é negativo
spark.sql("""
    SELECT COUNT(*) AS dias_negativos
    FROM judicial.silver.movimentos
    WHERE dias_desde_ajuizamento < 0
""").show()


In [0]:
# Timeline de um processo para conferir visualmente
spark.sql("""
    SELECT
        numero_processo,
        nome_movimento,
        data_movimento,
        dias_desde_ajuizamento,
        dias_desde_ultimo_movimento,
        is_ultimo_movimento
    FROM judicial.silver.movimentos
    WHERE numero_processo = (SELECT numero_processo FROM judicial.silver.movimentos LIMIT 1)
    ORDER BY data_movimento
""").show(truncate=False)
